# Phase 5 — RAG Knowledge Base (Gemini embeddings)

> **Gemini-only project.** Every cell here uses Google Gemini (chat + embeddings) via `langchain-google-genai`. The only key the core needs is `GOOGLE_API_KEY`. This notebook is a **scaffold** — run it top-to-bottom *after* Claude Code finishes this phase. If a cell references a module that doesn't exist yet, that phase hasn't been built.

**Goal:** ingest SEC filings for **individual-stock** tickers, embed with Gemini, retrieve with citations.

**👤 You provide:** `SEC_USER_AGENT` (name + email) in `.env` — EDGAR rejects requests without it.
**Data fact:** ETFs/cash (VTI, BND, QQQ, CASH...) do NOT file 10-Ks — they are skipped. So CLT-001/003/004 legitimately have no filings.


In [ ]:
# --- Setup: load .env and make the project importable ---
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
from dotenv import load_dotenv
load_dotenv('../.env')

# This project is Gemini-only. Confirm the one required key is present.
assert os.getenv('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY in .env (see 00_prerequisites_and_setup.md)'
print('GOOGLE_API_KEY loaded:', bool(os.getenv('GOOGLE_API_KEY')))
print('SEC_USER_AGENT   :', os.getenv('SEC_USER_AGENT', '(set before Phase 5)'))

In [ ]:
assert os.getenv('SEC_USER_AGENT'), 'Set SEC_USER_AGENT (name + email) in .env for Phase 5'

## 1. Ingest a couple of individual-stock tickers

In [ ]:
from app.knowledge.ingestion import ingest
ingest(tickers=['NVDA', 'MSFT'], limit=5)  # idempotent; skips ETFs automatically

## 2. Collection sizes

In [ ]:
from app.knowledge.vector_store import get_vector_store
vs = get_vector_store()
for c in ['sec_filings', 'news_archive', 'research_notes']:
    print(c, '->', vs.count(c))

## 3. Grounded retrieval with a citation (Gemini query embedding)

In [ ]:
from app.knowledge.retriever import Retriever
r = Retriever()
hits = r.query('data center demand', filters={'ticker': 'NVDA'}, k=3)
for content, cite in hits:
    print(f"[{cite.get('form')} {cite.get('ticker')} {cite.get('filing_date')}] {content[:120]}...")

## ✅ Acceptance check

In [ ]:
aapl = r.query('anything', filters={'ticker': 'AAPL'}, k=3)  # not ingested
assert len(hits) > 0 and len(aapl) == 0, 'NVDA grounded, AAPL absent'
print('Phase 5 acceptance: PASS (no OpenAI key was ever requested)')